# Data Analytics Lab Sheet 5: Interactive Dashboard Preparation and KPI Analysis

**Name:** ANURAG PARMAR  
**Roll No.:** 8  
**Section:** B  
**CU ID:** CU24250076  
**Program:** B.Tech CSE  

## Aim
Prepare a Superstore dataset, calculate dashboard KPIs, and build Python visuals and filterable summaries that can guide a Tableau or Power BI dashboard.

> Run cells from top to bottom. Dataset sources and any synthetic-data notes are documented in the relevant sections.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

## 2. Load and prepare data

In [ ]:
url = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"
df = pd.read_csv(url, encoding="latin1")
for col in ["Order Date", "Ship Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Calculate key performance indicators

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique() if "Order ID" in df.columns else len(df)
total_quantity = df["Quantity"].sum()
average_discount = df["Discount"].mean()
profit_margin = (total_profit / total_sales * 100) if total_sales else np.nan

kpis = pd.DataFrame({
    "KPI": ["Total Sales", "Total Profit", "Unique Orders", "Quantity Sold", "Average Discount", "Profit Margin (%)"],
    "Value": [total_sales, total_profit, total_orders, total_quantity, average_discount, profit_margin]
})
display(kpis)

## 4. Sales and profit by category/region

In [ ]:
category_summary = df.groupby("Category", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
region_summary = df.groupby("Region", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
display(category_summary.sort_values("Sales", ascending=False))
display(region_summary.sort_values("Profit", ascending=False))

category_summary.set_index("Category")[["Sales", "Profit"]].plot(kind="bar", figsize=(9, 5))
plt.title("Sales and Profit by Category")
plt.ylabel("Amount")
plt.tight_layout()
plt.show()

region_summary.set_index("Region")[["Sales", "Profit"]].plot(kind="bar", figsize=(9, 5))
plt.title("Sales and Profit by Region")
plt.tight_layout()
plt.show()

## 5. Monthly trend and top products

In [ ]:
if "Order Date" in df.columns:
    monthly = df.dropna(subset=["Order Date"]).set_index("Order Date").resample("MS")[["Sales", "Profit"]].sum()
    monthly.plot(figsize=(12, 5), marker="o")
    plt.title("Monthly Sales and Profit")
    plt.tight_layout()
    plt.show()

top_products = (df.groupby("Product Name", as_index=False)["Sales"].sum()
                .sort_values("Sales", ascending=False).head(10))
display(top_products)
sns.barplot(data=top_products, y="Product Name", x="Sales")
plt.title("Top 10 Products by Sales")
plt.tight_layout()
plt.show()

## 6. Dashboard filter example

In [ ]:
# Change these values to simulate dashboard filters.
selected_region = "West"  # e.g. "East", "Central", "South"
filtered = df[df["Region"] == selected_region].copy()
print("Selected region:", selected_region)
print("Sales:", round(filtered["Sales"].sum(), 2))
print("Profit:", round(filtered["Profit"].sum(), 2))
display(filtered.head())
filtered.to_csv("superstore_dashboard_data.csv", index=False)
print("Exported superstore_dashboard_data.csv for Tableau/Power BI.")